# Customer Churn - End-to-End Snowflake ML Lifecycle

1. Snapshot `CUSTOMER_CHURN_DB.CHURN.CUSTOMER_CHURN_TRAINING` into a versioned Snowflake **Dataset** and load it into pandas
2. Random stratified train/valid split
3. Train Logistic Regression, Elastic Net, Random Forest and XGBoost, logging params and metrics to a timestamped Snowflake **Experiment**
4. Compare ROC/AUC across models
5. Register the top model (by validation AUC) in the **Model Registry** (`CUSTOMER_CHURN_DB.CHURN`) with **lineage** back to the training Dataset and source table
6. Score with the registered model directly from **SQL**
7. Verify lineage: table -> dataset -> model

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.registry import Registry
from snowflake.snowpark.context import get_active_session
from xgboost import XGBClassifier

session = get_active_session()

DATABASE = "CUSTOMER_CHURN_DB"
SCHEMA = "CHURN"
SOURCE_TABLE = f"{DATABASE}.{SCHEMA}.CUSTOMER_CHURN_TRAINING"

# ACCOUNTADMIN owns CUSTOMER_CHURN_DB.CHURN (needs CREATE EXPERIMENT / CREATE MODEL)
session.use_role("ACCOUNTADMIN")
session.use_warehouse("COMPUTE_WH")
session.use_database(DATABASE)
session.use_schema(SCHEMA)

# One timestamp per notebook run keeps experiment / model version names unique
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Role: {session.get_current_role()} | Warehouse: {session.get_current_warehouse()}")
print(f"Context: {session.get_current_database()}.{session.get_current_schema()} | RUN_TS: {RUN_TS}")

## 1. Load data

In [ ]:
from snowflake.ml.dataset import create_from_dataframe

FEATURES = [
    "LOGINS_LAST_12_MONTHS", "LOGINS_LAST_24_MONTHS",
    "TRANSACTIONS_LAST_12_MONTHS", "TRANSACTIONS_LAST_24_MONTHS",
    "CUSTOMER_AGE",
]
TARGET = "CHURN"

# Immutable, versioned snapshot of the training data -> captures lineage TABLE -> DATASET
DATASET_NAME = "CUSTOMER_CHURN_TRAINING_DS"
DATASET_VERSION = f"V_{RUN_TS}"
train_ds = create_from_dataframe(
    session,
    DATASET_NAME,
    DATASET_VERSION,
    input_dataframe=session.table(SOURCE_TABLE),
    comment=f"Training snapshot of {SOURCE_TABLE}",
)
print(f"Dataset: {DATABASE}.{SCHEMA}.{DATASET_NAME} version {DATASET_VERSION}")

churn_pdf = train_ds.read.to_pandas()

X = churn_pdf[FEATURES].astype("float64")  # float keeps NULLs as NaN
y = churn_pdf[TARGET].astype(int)

print(f"Rows: {len(churn_pdf):,} | Churn rate: {y.mean():.3f}")
print("Null counts:\n", X.isna().sum())
churn_pdf.head()

## 2. Train / validation split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape} churn={y_train.mean():.3f}")
print(f"Valid: {X_valid.shape} churn={y_valid.mean():.3f}")

## 3. Experiment: train and log 4 models

In [ ]:
EXPERIMENT_NAME = f"CHURN_MODELS_{RUN_TS}"

exp = ExperimentTracking(session=session, database_name=DATABASE, schema_name=SCHEMA)
exp.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {DATABASE}.{SCHEMA}.{EXPERIMENT_NAME}")

In [ ]:
# l1_ratio=0 -> L2 penalty; 0 < l1_ratio < 1 -> elastic net (sklearn >= 1.8 style, no `penalty` arg)
lr_params = {"l1_ratio": 0.0, "C": 1.0, "solver": "lbfgs", "max_iter": 1000}
enet_params = {"l1_ratio": 0.5, "C": 1.0, "solver": "saga", "max_iter": 1000}
rf_params = {"n_estimators": 200, "max_depth": 12, "min_samples_leaf": 20, "n_jobs": -1, "random_state": 42}
xgb_params = {
    "n_estimators": 300, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.8,
    "colsample_bytree": 0.8, "tree_method": "hist", "n_jobs": -1, "random_state": 42,
}

candidates = {
    "LOGISTIC_REGRESSION": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(**lr_params)),
        ]),
        lr_params,
    ),
    "ELASTIC_NET": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(**enet_params)),
        ]),
        enet_params,
    ),
    "RANDOM_FOREST": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(**rf_params)),
        ]),
        rf_params,
    ),
    # XGBoost handles NaN natively, no imputer needed
    "XGBOOST": (XGBClassifier(**xgb_params), xgb_params),
}

In [ ]:
results = {}

for name, (model, params) in candidates.items():
    with exp.start_run(f"{name}_{RUN_TS}"):
        exp.log_params({"model_type": name, **params})

        model.fit(X_train, y_train)
        p_train = model.predict_proba(X_train)[:, 1]
        p_valid = model.predict_proba(X_valid)[:, 1]
        y_pred = (p_valid >= 0.5).astype(int)

        metrics = {
            "valid_auc": roc_auc_score(y_valid, p_valid),
            "train_auc": roc_auc_score(y_train, p_train),
            "valid_accuracy": accuracy_score(y_valid, y_pred),
            "valid_precision": precision_score(y_valid, y_pred),
            "valid_recall": recall_score(y_valid, y_pred),
            "valid_f1": f1_score(y_valid, y_pred),
            "valid_log_loss": log_loss(y_valid, p_valid),
        }
        exp.log_metrics(metrics)

    results[name] = {"model": model, "p_valid": p_valid, **metrics}
    print(f"{name:<20} valid AUC={metrics['valid_auc']:.4f}  train AUC={metrics['train_auc']:.4f}")

## 4. Compare models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_valid, r["p_valid"])
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {r['valid_auc']:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.5)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"Validation ROC - {EXPERIMENT_NAME}")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
leaderboard = (
    pd.DataFrame({n: {k: v for k, v in r.items() if k not in ("model", "p_valid")} for n, r in results.items()})
    .T.sort_values("valid_auc", ascending=False)
)
best_name = leaderboard.index[0]
best_model = results[best_name]["model"]
print(f"Best model: {best_name} (valid AUC = {leaderboard.loc[best_name, 'valid_auc']:.4f})")
leaderboard

## 5. Register the top model

In [ ]:
from snowflake.snowpark.functions import col
from snowflake.snowpark.types import DoubleType

MODEL_NAME = "CUSTOMER_CHURN_MODEL"
VERSION_NAME = f"V_{RUN_TS}"

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

# Snowpark DataFrame backed by the training Dataset -> captures lineage DATASET -> MODEL
# (a pandas sample would register the model with no upstream lineage)
lineage_sample = (
    train_ds.read.to_snowpark_dataframe()
    .select([col(f).cast(DoubleType()).alias(f) for f in FEATURES])
    .limit(100)
)

mv = reg.log_model(
    best_model,
    model_name=MODEL_NAME,
    version_name=VERSION_NAME,
    sample_input_data=lineage_sample,
    target_platforms=["WAREHOUSE"],
    metrics={
        **{k: float(v) for k, v in leaderboard.loc[best_name].items()},
        "model_type": best_name,
        "experiment": EXPERIMENT_NAME,
        "training_dataset": f"{DATABASE}.{SCHEMA}.{DATASET_NAME}/{DATASET_VERSION}",
    },
    comment=f"Churn classifier ({best_name}) - best valid AUC from experiment {EXPERIMENT_NAME}",
)

# Make this version the default so unversioned calls use it
reg.get_model(MODEL_NAME).default = VERSION_NAME

print(f"Registered {DATABASE}.{SCHEMA}.{mv.model_name} version {mv.version_name}")
mv.show_functions()

## 6. Score with SQL

The registered model is callable from any SQL query with `MODEL(<name>, <version>)!PREDICT_PROBA(...)`. `{{VERSION_NAME}}` is pulled from the Python cell above.

In [ ]:
%%sql -r scored_sample
SELECT
    CUSTOMER_ID,
    SNAPSHOT_DATE,
    CHURN AS ACTUAL_CHURN,
    MODEL(CUSTOMER_CHURN_DB.CHURN.CUSTOMER_CHURN_MODEL, {{VERSION_NAME}})!PREDICT_PROBA(
        LOGINS_LAST_12_MONTHS::FLOAT,
        LOGINS_LAST_24_MONTHS::FLOAT,
        TRANSACTIONS_LAST_12_MONTHS::FLOAT,
        TRANSACTIONS_LAST_24_MONTHS::FLOAT,
        CUSTOMER_AGE::FLOAT
    ):output_feature_1::FLOAT AS CHURN_PROBABILITY
FROM CUSTOMER_CHURN_DB.CHURN.CUSTOMER_CHURN_TRAINING SAMPLE (10000 ROWS)
ORDER BY CHURN_PROBABILITY DESC;

In [ ]:
%%sql -r scoring_summary
SELECT
    COUNT(*)                                                     AS SCORED_ROWS,
    AVG(ACTUAL_CHURN)                                            AS ACTUAL_CHURN_RATE,
    AVG(CHURN_PROBABILITY)                                       AS AVG_PREDICTED_PROBABILITY,
    AVG(IFF(CHURN_PROBABILITY >= 0.5, 1, 0))                     AS PREDICTED_CHURN_RATE,
    AVG(IFF(IFF(CHURN_PROBABILITY >= 0.5, 1, 0) = ACTUAL_CHURN, 1, 0)) AS ACCURACY_AT_0_5
FROM {{scored_sample}};

## 7. Lineage: source table -> Dataset -> Model

Also viewable in Snowsight under **AI & ML -> Models -> CUSTOMER_CHURN_MODEL -> Lineage**.

In [ ]:
%%sql -r model_lineage
SELECT
    DISTANCE,
    SOURCE_OBJECT_DOMAIN,
    SOURCE_OBJECT_DATABASE || '.' || SOURCE_OBJECT_SCHEMA || '.' || SOURCE_OBJECT_NAME AS SOURCE_OBJECT,
    SOURCE_OBJECT_VERSION,
    TARGET_OBJECT_DOMAIN,
    TARGET_OBJECT_DATABASE || '.' || TARGET_OBJECT_SCHEMA || '.' || TARGET_OBJECT_NAME AS TARGET_OBJECT,
    TARGET_OBJECT_VERSION
FROM TABLE(
    SNOWFLAKE.CORE.GET_LINEAGE(
        'CUSTOMER_CHURN_DB.CHURN.CUSTOMER_CHURN_MODEL',
        'MODEL',
        'UPSTREAM',
        3,
        '{{VERSION_NAME}}'
    )
)
ORDER BY DISTANCE;

In [ ]:
print(f"Upstream of model {MODEL_NAME}/{VERSION_NAME}:")
for node in mv.lineage(direction="upstream"):
    print(f"  {node._lineage_node_domain}: {node._lineage_node_name} {node._lineage_node_version or ''}")

print(f"\nUpstream of dataset {DATASET_NAME}/{DATASET_VERSION}:")
for node in train_ds.lineage(direction="upstream"):
    print(f"  {node._lineage_node_domain}: {node._lineage_node_name}")